In [1]:
## Bước 1: Khởi tạo SparkSession và nạp dữ liệu đã làm sạch từ HDFS
import findspark
findspark.init()
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql.types import IntegerType
spark = SparkSession.builder \
    .appName("Train_Model_Du_Doan_Mua_Hang") \
    .master("spark://26.37.93.102:7077") \
    .config("spark.executor.memory", "4g") \
    .config("spark.driver.memory", "2g") \
    .getOrCreate()

spark.sparkContext.setLogLevel("ERROR")
print("Đã tạo thành công kết nối SparkSession!")
df = spark.read.parquet("hdfs://master:9000/data/test_cleaned.parquet")
print(f"Tổng số dòng dữ liệu sẵn sàng cho mô hình: {df.count():,}")

Đã tạo thành công kết nối SparkSession!
Tổng số dòng dữ liệu sẵn sàng cho mô hình: 2,711,122


In [2]:
## Bước 1: Khám phá dữ liệu và kiểm tra phân phối nhãn
print("=" * 60)
print("THÔNG TIN SCHEMA")
print("=" * 60)
df.printSchema()

print("\n" + "=" * 60)
print("PHÂN PHỐI LOẠI SỰ KIỆN (event_type)")
print("=" * 60)
df.groupBy("event_type").count().orderBy(F.desc("count")).show()

print("\n" + "=" * 60)
print("THỐNG KÊ MÔ TẢ CỘT SỐ")
print("=" * 60)
df.select("price", "ts_hour", "ts_weekday").describe().show()

THÔNG TIN SCHEMA
root
 |-- product_id: string (nullable = true)
 |-- event_time: string (nullable = true)
 |-- event_type: string (nullable = true)
 |-- brand: string (nullable = true)
 |-- price: double (nullable = true)
 |-- user_id: string (nullable = true)
 |-- user_session: string (nullable = true)
 |-- target: long (nullable = true)
 |-- cat_0: string (nullable = true)
 |-- cat_1: string (nullable = true)
 |-- cat_2: string (nullable = true)
 |-- timestamp: timestamp (nullable = true)
 |-- ts_hour: short (nullable = true)
 |-- ts_minute: short (nullable = true)
 |-- ts_weekday: short (nullable = true)
 |-- ts_day: short (nullable = true)
 |-- ts_month: short (nullable = true)
 |-- ts_year: short (nullable = true)


PHÂN PHỐI LOẠI SỰ KIỆN (event_type)
+----------+-------+
|event_type|  count|
+----------+-------+
|      cart|1751346|
|  purchase| 959776|
+----------+-------+


THỐNG KÊ MÔ TẢ CỘT SỐ
+-------+------------------+------------------+------------------+
|summary|       

In [3]:
## Bước 2: Lọc dữ liệu và tạo nhãn
# Lọc chỉ giữ sự kiện liên quan đến hành trình mua hàng
df_filtered = df.filter(
    (F.col("event_type").isin(["view", "cart", "purchase"])) &
    (F.col("price") > 0) &
    F.col("price").isNotNull() &
    F.col("brand").isNotNull() &
    F.col("cat_0").isNotNull() &
    F.col("ts_hour").isNotNull() &
    F.col("ts_weekday").isNotNull()
)

# Tạo nhãn nhị phân: 1 = mua hàng, 0 = chưa mua
df_labeled = df_filtered.withColumn(
    "label",
    F.when(F.col("event_type") == "purchase", 1).otherwise(0).cast(IntegerType())
)

total = df_labeled.count()
pos = df_labeled.filter(F.col("label") == 1).count()
neg = total - pos

print(f"Tổng dòng sau lọc    : {total:,}")
print(f"Label = 1 (purchase) : {pos:,} ({pos/total*100:.2f}%)")
print(f"Label = 0 (no buy)   : {neg:,} ({neg/total*100:.2f}%)")
print(f"Tỷ lệ mất cân bằng   : 1:{neg//pos if pos > 0 else 'N/A'}")

Tổng dòng sau lọc    : 2,711,122
Label = 1 (purchase) : 959,776 (35.40%)
Label = 0 (no buy)   : 1,751,346 (64.60%)
Tỷ lệ mất cân bằng   : 1:1


In [4]:
## Bước 3: Tạo đặc trưng mới
df_feat = df_labeled \
    .withColumn(
        "is_golden_hour",
        F.when(
            (F.col("ts_hour").between(10, 12)) | (F.col("ts_hour").between(20, 22)), 1
        ).otherwise(0).cast(IntegerType())
    ) \
    .withColumn(
        "is_weekend",
        F.when(F.col("ts_weekday").isin([5, 6]), 1).otherwise(0).cast(IntegerType())
    ) \
    .withColumn(
        "price_log",
        F.log1p(F.col("price"))          # log(1 + price) — giảm độ lệch phân phối giá
    ) \
    .withColumn(
        "price_bucket",
        F.when(F.col("price") < 10, "low")
         .when(F.col("price") < 50, "mid")
         .when(F.col("price") < 200, "high")
         .otherwise("premium")
    ) \
    .withColumn(
        "session_part",
        F.when(F.col("ts_hour").between(0, 5), "night")
         .when(F.col("ts_hour").between(6, 11), "morning")
         .when(F.col("ts_hour").between(12, 17), "afternoon")
         .otherwise("evening")
    )

print("Các đặc trưng sau Feature Engineering:")
df_feat.select(
    "price", "price_log", "price_bucket", "ts_hour", "session_part",
    "ts_weekday", "is_weekend", "is_golden_hour", "brand", "cat_0", "label"
).show(5, truncate=False)

Các đặc trưng sau Feature Engineering:
+------------------+-----------------+------------+-------+------------+----------+----------+--------------+-------+------------+-----+
|price             |price_log        |price_bucket|ts_hour|session_part|ts_weekday|is_weekend|is_golden_hour|brand  |cat_0       |label|
+------------------+-----------------+------------+-------+------------+----------+----------+--------------+-------+------------+-----+
|119.44000244140625|4.791151723884259|high        |16     |afternoon   |2         |0         |0             |delta  |construction|0    |
|119.44000244140625|4.791151723884259|high        |16     |afternoon   |2         |0         |0             |delta  |construction|0    |
|56.369998931884766|4.04952149999577 |high        |14     |afternoon   |4         |0         |0             |dam    |apparel     |0    |
|28.200000762939453|3.374168735402299|mid         |8      |morning     |6         |1         |0             |crucial|apparel     |1    |
|2

In [5]:
## Bước 4: Xây dựng Spark ML Pipeline
from pyspark.ml import Pipeline
from pyspark.ml.feature import (
    StringIndexer, OneHotEncoder, VectorAssembler, StandardScaler
)

# ── 1. StringIndexer: chuỗi → số nguyên ──────────────────────────────────────
cat_cols        = ["brand", "cat_0", "price_bucket", "session_part"]
indexed_cols    = [c + "_idx"  for c in cat_cols]
encoded_cols    = [c + "_ohe"  for c in cat_cols]

indexers = [
    StringIndexer(inputCol=c, outputCol=c + "_idx", handleInvalid="skip")
    for c in cat_cols
]

# ── 2. OneHotEncoder: số nguyên → vector nhị phân ────────────────────────────
encoder = OneHotEncoder(
    inputCols=indexed_cols,
    outputCols=encoded_cols,
    dropLast=True
)

# ── 3. VectorAssembler: gom tất cả đặc trưng thành 1 vector ──────────────────
numeric_cols = ["price_log", "ts_hour", "ts_weekday", "is_weekend", "is_golden_hour"]
all_feature_cols = numeric_cols + encoded_cols

assembler = VectorAssembler(
    inputCols=all_feature_cols,
    outputCol="features_raw",
    handleInvalid="skip"
)

# ── 4. StandardScaler: đưa về cùng thang đo ──────────────────────────────────
scaler = StandardScaler(
    inputCol="features_raw",
    outputCol="features",
    withMean=False,   # False vì OHE vector thưa (sparse)
    withStd=True
)

print("Đã định nghĩa các bước tiền xử lý:")
print(f"   Categorical  : {cat_cols}")
print(f"   Numeric      : {numeric_cols}")
print(f"   Tổng features: {len(all_feature_cols)} nhóm cột (trước OHE expand)")

Đã định nghĩa các bước tiền xử lý:
   Categorical  : ['brand', 'cat_0', 'price_bucket', 'session_part']
   Numeric      : ['price_log', 'ts_hour', 'ts_weekday', 'is_weekend', 'is_golden_hour']
   Tổng features: 9 nhóm cột (trước OHE expand)


In [6]:
## Bước 5: Train/ Test Split & xử lý mất cân bằng lớp
# Tách tập train/test theo tỷ lệ 80/20
train_df, test_df = df_feat.randomSplit([0.8, 0.2], seed=42)

# Tính classWeight để cân bằng lớp trong Random Forest
train_count   = train_df.count()
pos_count     = train_df.filter(F.col("label") == 1).count()
neg_count     = train_count - pos_count
balance_ratio = neg_count / pos_count if pos_count > 0 else 1.0

print(f"Train size : {train_count:,}")
print(f"Test  size : {test_df.count():,}")
print(f"Balance ratio (neg/pos): {balance_ratio:.2f}  → dùng làm classWeight")

# Gắn cột classWeight để hỗ trợ các thuật toán có tham số weightCol
train_weighted = train_df.withColumn(
    "classWeight",
    F.when(F.col("label") == 1, balance_ratio).otherwise(1.0)
)

Train size : 2,169,436
Test  size : 541,686
Balance ratio (neg/pos): 1.82  → dùng làm classWeight


In [7]:
## Bước 6: Huấn luyện mô hình Random Forest Classifier
from pyspark.ml.classification import RandomForestClassifier, GBTClassifier

# ── Random Forest ─────────────────────────────────────────────────────────────
rf = RandomForestClassifier(
    labelCol="label",
    featuresCol="features",
    numTrees=100,
    maxDepth=8,
    minInstancesPerNode=10,
    featureSubsetStrategy="sqrt",
    seed=42
)

# ── Pipeline hoàn chỉnh: tiền xử lý + mô hình ────────────────────────────────
pipeline_rf = Pipeline(stages=indexers + [encoder, assembler, scaler, rf])

print("⏳ Đang huấn luyện Random Forest...")
model_rf = pipeline_rf.fit(train_weighted)
print("✅ Huấn luyện Random Forest hoàn tất!")

⏳ Đang huấn luyện Random Forest...
✅ Huấn luyện Random Forest hoàn tất!


In [8]:
## Bước 7: Huấn luyện mô hình bổ sung GBT
from pyspark.ml.classification import GBTClassifier
from pyspark.ml import Pipeline

## Bước 7: Huấn luyện mô hình bổ sung GBT
gbt = GBTClassifier(
    labelCol="label",
    featuresCol="features",
    maxIter=50,
    maxDepth=6,
    stepSize=0.1,
    subsamplingRate=0.8,
    seed=42
)

pipeline_gbt = Pipeline(stages=indexers + [encoder, assembler, scaler, gbt])

print("⏳ Đang nạp dữ liệu vào RAM (Caching) để tăng tốc GBT...")
# --- ÁP DỤNG GIẢI PHÁP 1 ---
# 1. Ra lệnh cho Spark lưu tập dữ liệu này vào bộ nhớ đệm
train_weighted.cache()

# 2. Gọi một "Action" (như count) để ép Spark thực hiện việc cache ngay lập tức.
# Nếu không có dòng này, Spark (với bản chất lười biếng) sẽ đợi đến lúc gọi hàm .fit() mới bắt đầu cache.
total_rows = train_weighted.count()
print(f"✅ Đã cache xong {total_rows:,} dòng dữ liệu vào RAM!")
# ---------------------------

print("⏳ Đang huấn luyện GBT (sẽ nhanh hơn rất nhiều)...")
model_gbt = pipeline_gbt.fit(train_weighted)
print("✅ Huấn luyện GBT hoàn tất!")

⏳ Đang nạp dữ liệu vào RAM (Caching) để tăng tốc GBT...
✅ Đã cache xong 2,169,436 dòng dữ liệu vào RAM!
⏳ Đang huấn luyện GBT (sẽ nhanh hơn rất nhiều)...


ERROR:root:KeyboardInterrupt while sending command.
Traceback (most recent call last):
  File "D:\BigData\SparkN\spark-3.5.8-bin-hadoop3\python\lib\py4j-0.10.9.7-src.zip\py4j\java_gateway.py", line 1038, in send_command
    response = connection.send_command(command)
               ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "D:\BigData\SparkN\spark-3.5.8-bin-hadoop3\python\lib\py4j-0.10.9.7-src.zip\py4j\clientserver.py", line 511, in send_command
    answer = smart_decode(self.stream.readline()[:-1])
                          ^^^^^^^^^^^^^^^^^^^^^^
  File "D:\BigData\Python\Lib\socket.py", line 720, in readinto
    return self._sock.recv_into(b)
           ^^^^^^^^^^^^^^^^^^^^^^^
KeyboardInterrupt


KeyboardInterrupt: 